# Bronze Layer: Customer Info (Auto Loader)

**Pattern**: Auto Loader (Streaming Ingestion)  
**Trigger**: AvailableNow (Incremental Batch)  

This notebook uses Spark Auto Loader to ingest data incrementally. It is compatible with Databricks Free Edition.

In [0]:
from pyspark.sql.functions import current_timestamp

# Paths
source_path = "/Volumes/workspace/bronze/raw_sources/source_crm/"
target_table = "workspace.bronze.crm_cust_info_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/crm_cust_info"
schema_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/crm_cust_info_schema"

# 1. Read stream using Auto Loader
df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("pathGlobFilter", "cust_info.csv")
    .load(source_path)
    .withColumn("ingestion_timestamp", current_timestamp())
)

# 2. Write stream with AvailableNow trigger (Batch Streaming)
print(f"Starting incremental ingestion for {target_table}...")

query = (
    df_stream
    .drop("_rescued_data") # Drop rescued data to match existing table schema
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
print(f"✓ Ingestion complete. Total records: {spark.table(target_table).count():,}")